# dMRI Region Patch Stitching

Within a selected brain region, validate the spatial continuity of the embeddings after **global** dimensionality reduction.

**Workflow:** Sample N nodes from the entire brain region → apply global dimensionality reduction using NOFE / PCA / t-SNE / UMAP  
→ split the nodes into A / B sides based on the median coordinate → compute the normalised embedding distance across the boundary from A→B and B→A.


In [ ]:
import sys, json, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap as umap_lib

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(Path.cwd().parent / "scripts"))
from utilities import KernelNN, setup_device
from utilities_dMRI import dMRI_Graph_Generator

device = setup_device()
print(f"Using device: {device}")


def mean_dist(x: torch.Tensor) -> torch.Tensor:
    N = x.shape[0]
    return torch.cdist(x, x).triu(diagonal=1).sum() / (N * (N - 1) / 2)


def get_region_mask(Generator, region: str) -> torch.Tensor:
    y = Generator.normed_coords[:, 1]
    z = Generator.normed_coords[:, 2]
    masks = {
        "frontal":     y > 0.60,
        "parietal":    (y > 0.35) & (y < 0.85) & (z > 0.62),
        "temporal":    (y > 0.20) & (y < 0.70) & (z < 0.55),
        "occipital":   y < 0.25,
        "subcortical": (y > 0.35) & (y < 0.65) & (z > 0.28) & (z < 0.58),
    }
    if region not in masks:
        raise ValueError(f"Unknown region: {region}")
    return masks[region]


def sample_region_graph(Generator, region_mask, N, k, perplexity, feature_mode):
    """Sample N nodes from the full region (no spatial split)."""
    orig_da, orig_coords = Generator.da, Generator.normed_coords
    Generator.da = orig_da[region_mask]
    Generator.normed_coords = orig_coords[region_mask]
    graph = Generator.sample_graph(
        N=min(N, Generator.da.shape[0]), k=k, perplexity=perplexity,
        sampling_method="random", gen_feature_mode=feature_mode,
    )
    Generator.da, Generator.normed_coords = orig_da, orig_coords
    return graph.to(device)


def preprocess_for_baselines(X: np.ndarray) -> np.ndarray:
    p995 = np.percentile(X, 99.5, axis=0)
    return (np.clip(X, 0, p995) - X.mean(0)) / (X.std(0) + 1e-8)


def load_dMRI_model(model_id: str, model_type: str = "regular"):
    mp = project_root / "dMRI" / "trained_models" / model_type / f"{model_id}.pth"
    cp = project_root / "dMRI" / "trained_models" / model_type / "logs" / "configs" / f"{model_id}.json"
    with open(cp) as f:
        config = json.load(f)
    model = KernelNN(
        width=config["width"], ker_width=config["ker_width"], depth=config["depth"],
        ker_in=config["edge_attr"], in_width=config["node_features"], out_width=config["out_width"],
    ).to(device)
    model.load_state_dict(torch.load(mp, map_location=device))
    model.eval()
    return model, config

---
### Load dMRI model

In [ ]:
model_id = "base_2026-04-29_t11-16-23"
model, config = load_dMRI_model(model_id)
k            = config["k"]
perplexity   = config["perplexity"]
feature_mode = config["feature_mode"]
print(f"Model: {model_id}  k={k}  perplexity={perplexity}")

---
### Configuration

In [ ]:
# region = "frontal" / "parietal" / "temporal" / "occipital"
# region     = "parietal"
region     = "temporal"
split_axis = 0        # 0=X (L-R), 1=Y (ant-post), 2=Z (sup-inf)

N           = 5000    # nodes sampled from the whole region
edge_margin = 0.05    # half-width of border zone in normalised coords

axis_labels = ["x (L-R)", "y (ant-post)", "z (sup-inf)"]
axis_label  = axis_labels[split_axis]
v_axis      = 2 if split_axis != 2 else 1
v_label     = axis_labels[v_axis]

data_root = project_root / "dMRI" / "data"
subjects = sorted([
    d for d in os.listdir(data_root)
    if d.isdigit() and (data_root / d / "T1w" / "Diffusion_7T").exists()
])
test_subjects = subjects[-5:] if len(subjects) >= 5 else subjects
methods = ["nofe", "pca", "tsne", "umap"]
print(f"Region: {region}  split: {axis_label}  N={N}")
print(f"Test subjects: {test_subjects}")

---
## Visual Check — Region Embedding (one subject)

Sample N nodes from the brain region, perform global dimensionality reduction, and visualize them as a scatter plot along `split_axis` vs `v_axis`.

RGB = globally normalized first three embedding dimensions.  
Dashed line = `split_point` (median coordinate).

In [ ]:
Generator = dMRI_Graph_Generator(str(data_root))
subj_demo = test_subjects[-1]
print(f"Loading {subj_demo} ...")
Generator.load_data(str(data_root / subj_demo / "T1w" / "Diffusion_7T") + "/")

r_mask_demo = get_region_mask(Generator, region)
g_demo = sample_region_graph(
    Generator, r_mask_demo, N, k, perplexity, feature_mode
)
print(f"Sampled: {g_demo.normed_coords.shape[0]} nodes")

X_demo = preprocess_for_baselines(g_demo.normed_values.detach().cpu().numpy())

print("NOFE  ...", end="\r")
emb_nofe_d = model(g_demo).detach().cpu().numpy()
if torch.backends.mps.is_available(): torch.mps.empty_cache()

print("PCA   ...", end="\r")
emb_pca_d = PCA(n_components=3).fit_transform(X_demo)

print("t-SNE ...", end="\r")
emb_tsne_d = TSNE(
    n_components=3, perplexity=min(30, g_demo.normed_coords.shape[0] - 1),
    learning_rate="auto", random_state=42, method="exact"
).fit_transform(X_demo)

print("UMAP  ...", end="\r")
emb_umap_d = umap_lib.UMAP(n_components=3, random_state=42).fit_transform(X_demo)
print("Done.")

In [ ]:
def make_rgb(emb: np.ndarray) -> np.ndarray:
    """Robust RGB normalisation: clip 1-99 percentile per channel before min-max."""
    p1  = np.percentile(emb, 1,  axis=0)
    p99 = np.percentile(emb, 99, axis=0)
    emb = np.clip(emb, p1, p99)
    mn, mx = emb.min(0), emb.max(0)
    return (emb - mn) / (mx - mn + 1e-8)


coords_demo = g_demo.normed_coords.detach().cpu().numpy()
h_vals  = coords_demo[:, split_axis]
v_vals  = coords_demo[:, v_axis]
sp_demo = float(np.median(h_vals))

embeddings_demo = {
    "NOFE":  emb_nofe_d,
    "PCA":   emb_pca_d,
    "t-SNE": emb_tsne_d,
    "UMAP":  emb_umap_d,
}

fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (name, emb) in zip(axes.flat, embeddings_demo.items()):
    ax.scatter(h_vals, v_vals, c=make_rgb(emb), s=4, alpha=0.7, rasterized=True)
    ax.axvline(sp_demo, color="red", lw=0.9, ls="--", alpha=0.8, label="split")
    ax.set_title(name, fontsize=13)
    ax.set_xlabel(axis_label)
    ax.set_ylabel(v_label)
    ax.legend(fontsize=7)

plt.suptitle(
    f"Region: {region}  |  Subject {subj_demo}\n"
    "RGB = embedding dims 0-1-2  |  red dashed = split_point",
    fontsize=11
)
plt.tight_layout()
plt.savefig(Path.cwd() / f"region_{region}_visual.png", dpi=150)
plt.show()
print(f"Saved → region_{region}_visual.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter


def make_rgb(emb: np.ndarray) -> np.ndarray:
    """
    Robust RGB normalisation.
    Clip 1-99 percentile per embedding channel, then min-max to [0, 1].
    """
    emb = np.asarray(emb)

    p1 = np.percentile(emb, 1, axis=0)
    p99 = np.percentile(emb, 99, axis=0)

    emb_clip = np.clip(emb, p1, p99)
    mn = emb_clip.min(axis=0)
    mx = emb_clip.max(axis=0)

    return (emb_clip - mn) / (mx - mn + 1e-8)


def select_slice(coords, slice_axis, slice_value=None, slice_width=0.06):
    """
    Select a thin slab around one slice.

    For sparse sampled data, slice_width should not be too small.
    Try: 0.04, 0.06, 0.08.
    """
    vals = coords[:, slice_axis]

    if slice_value is None:
        slice_value = np.median(vals)

    mask = np.abs(vals - slice_value) <= slice_width
    return mask, slice_value, slice_width


def rgb_points_to_masked_image(
    x,
    y,
    rgb,
    resolution=240,
    method="linear",
    mask_factor=3.0,
    smooth_sigma=0.6,
    background=1.0,
):
    """
    Convert sparse 2D RGB points into a masked interpolated image.

    Improvements over simple nearest interpolation:
    1. Uses linear interpolation where possible.
    2. Falls back to nearest only inside valid tissue area.
    3. Masks pixels too far from sampled voxels.
    4. Optionally smooths the RGB image slightly.
    """

    x = np.asarray(x)
    y = np.asarray(y)
    rgb = np.asarray(rgb)

    points = np.stack([x, y], axis=1)

    x_min, x_max = x.min(), x.max()
    y_min, y_max = y.min(), y.max()

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution),
    )

    grid_points = np.stack([grid_x.ravel(), grid_y.ravel()], axis=1)

    # ------------------------------------------------------------
    # 1. Build a tissue mask based on nearest sampled voxel distance
    # ------------------------------------------------------------
    tree = cKDTree(points)

    # Distance from each grid pixel to nearest sampled voxel
    grid_dist, _ = tree.query(grid_points, k=1)
    grid_dist = grid_dist.reshape(resolution, resolution)

    # Estimate typical spacing among sampled points
    point_dist, _ = tree.query(points, k=2)
    typical_spacing = np.percentile(point_dist[:, 1], 75)

    valid_mask = grid_dist <= (mask_factor * typical_spacing)

    # ------------------------------------------------------------
    # 2. Interpolate RGB channels
    # ------------------------------------------------------------
    img_linear = np.zeros((resolution, resolution, 3))
    img_nearest = np.zeros((resolution, resolution, 3))

    for c in range(3):
        img_linear[..., c] = griddata(
            points,
            rgb[:, c],
            (grid_x, grid_y),
            method=method,
            fill_value=np.nan,
        )

        img_nearest[..., c] = griddata(
            points,
            rgb[:, c],
            (grid_x, grid_y),
            method="nearest",
            fill_value=np.nan,
        )

    # Use linear interpolation where available,
    # but fallback to nearest inside the valid tissue mask.
    image = img_linear.copy()
    nan_mask = np.isnan(image).any(axis=2)
    image[nan_mask] = img_nearest[nan_mask]

    # ------------------------------------------------------------
    # 3. Optional smoothing inside the valid region
    # ------------------------------------------------------------
    if smooth_sigma is not None and smooth_sigma > 0:
        smoothed = np.zeros_like(image)

        for c in range(3):
            channel = image[..., c].copy()
            channel[~valid_mask] = 0.0

            weight = valid_mask.astype(float)

            numerator = gaussian_filter(channel * weight, sigma=smooth_sigma)
            denominator = gaussian_filter(weight, sigma=smooth_sigma)

            smoothed[..., c] = numerator / (denominator + 1e-8)

        image = smoothed

    # ------------------------------------------------------------
    # 4. Set background outside tissue mask
    # ------------------------------------------------------------
    image = np.clip(image, 0, 1)
    image[~valid_mask] = background

    extent = [x_min, x_max, y_min, y_max]

    return image, extent, valid_mask


# ============================================================
# Main visualization
# ============================================================

coords_demo = g_demo.normed_coords.detach().cpu().numpy()

h_vals = coords_demo[:, split_axis]
v_vals = coords_demo[:, v_axis]

all_axes = {0, 1, 2}
slice_axis = list(all_axes - {split_axis, v_axis})[0]

sp_demo = float(np.median(h_vals))

embeddings_demo = {
    "NOFE": emb_nofe_d,
    "PCA": emb_pca_d,
    "t-SNE": emb_tsne_d,
    "UMAP": emb_umap_d,
}

slice_mask, slice_value, slice_width = select_slice(
    coords_demo,
    slice_axis=slice_axis,
    slice_value=None,
    slice_width=0.06,
)

print(f"Selected slice axis: {slice_axis}")
print(f"Slice value: {slice_value:.4f}")
print(f"Slice width: ±{slice_width:.4f}")
print(f"Voxels in slice: {slice_mask.sum()} / {len(slice_mask)}")


fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)

for ax, (name, emb) in zip(axes.flat, embeddings_demo.items()):

    rgb = make_rgb(emb)

    image, extent, valid_mask = rgb_points_to_masked_image(
        x=h_vals[slice_mask],
        y=v_vals[slice_mask],
        rgb=rgb[slice_mask],
        resolution=260,
        method="linear",
        mask_factor=3.0,
        smooth_sigma=0.6,
        background=1.0,
    )

    ax.imshow(
        image,
        extent=extent,
        origin="lower",
        aspect="equal",
        interpolation="nearest",
    )

    # Optional: overlay original sampled voxels lightly.
    # This helps show where the actual data points are.
    ax.scatter(
        h_vals[slice_mask],
        v_vals[slice_mask],
        c=rgb[slice_mask],
        s=2,
        alpha=0.35,
        linewidths=0,
        rasterized=True,
    )

    ax.axvline(
        sp_demo,
        color="red",
        lw=1.0,
        ls="--",
        alpha=0.9,
        label="split",
    )

    ax.set_title(name, fontsize=13)
    ax.set_xlabel(axis_label)
    ax.set_ylabel(v_label)
    ax.legend(fontsize=7, loc="upper right")

    ax.set_xlim(extent[0], extent[1])
    ax.set_ylim(extent[2], extent[3])


fig.suptitle(
    f"Region: {region}  |  Subject {subj_demo}\n"
    f"RGB = embedding dims 0-1-2  |  slice axis = {slice_axis}  |  red dashed = split point",
    fontsize=11,
)

out_path = Path.cwd() / f"region_{region}_slice_rgb_masked_optimized.png"
plt.savefig(out_path, dpi=200, bbox_inches="tight")
plt.show()

print(f"Saved → {out_path}")

---
### Border Zone Visualisation (AB and BA)

Following Figure 2 in the paper: the right boundary of patch A is **AB** (blue), and the left boundary of patch B is **BA** (red).

`edge_margin` controls the width of the boundary strip; cross-patch nearest-neighbor search is performed between AB and BA.

In [ ]:
# Use the demo graph's coordinates to illustrate the border zones
c_ax = coords_demo[:, split_axis]   # coordinate along split axis
sp_d = float(np.median(c_ax))

A_mask  =  c_ax <  sp_d
B_mask  =  c_ax >= sp_d
AB_mask = (c_ax <  sp_d) & (c_ax > sp_d - edge_margin)   # right edge of A
BA_mask = (c_ax >= sp_d) & (c_ax < sp_d + edge_margin)   # left  edge of B

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(c_ax[A_mask & ~AB_mask], coords_demo[A_mask & ~AB_mask, v_axis],
           c="#90CAF9", s=3, alpha=0.4, label="A (non-border)")
ax.scatter(c_ax[B_mask & ~BA_mask], coords_demo[B_mask & ~BA_mask, v_axis],
           c="#FFCDD2", s=3, alpha=0.4, label="B (non-border)")
ax.scatter(c_ax[AB_mask], coords_demo[AB_mask, v_axis],
           c="#1565C0", s=12, alpha=0.9, label=f"AB border  n={AB_mask.sum()}")
ax.scatter(c_ax[BA_mask], coords_demo[BA_mask, v_axis],
           c="#C62828", s=12, alpha=0.9, label=f"BA border  n={BA_mask.sum()}")

ax.axvline(sp_d,                color="black",   lw=1.5,  label="split_point")
ax.axvline(sp_d - edge_margin,  color="#1565C0", lw=1.0, ls="--", alpha=0.8,
           label=f"split − edge_margin ({edge_margin})")
ax.axvline(sp_d + edge_margin,  color="#C62828", lw=1.0, ls="--", alpha=0.8,
           label=f"split + edge_margin ({edge_margin})")

ax.set_xlabel(axis_label)
ax.set_ylabel(v_label)
ax.set_title(
    f"Border zones — region: {region}  split: {axis_label}\n"
    f"edge_margin = {edge_margin}  |  Subject {subj_demo}",
    fontsize=11
)
ax.legend(loc="upper right", fontsize=8, framealpha=0.8)
plt.tight_layout()
plt.savefig(Path.cwd() / f"region_{region}_borders.png", dpi=150)
plt.show()
print(f"AB border nodes: {AB_mask.sum()},  BA border nodes: {BA_mask.sum()}")

---
## Quantitative Evaluation

Sample N nodes from the entire brain region → perform global dimensionality reduction → split the nodes into A/B sides based on the median value along `split_axis`
→ identify boundary nodes (A-side nodes close to the boundary + B-side nodes close to the boundary) → compute the embedding distances from A→B and B→A.

In [ ]:
all_region_err   = {m: [] for m in methods}
all_neighbor_err = {m: [] for m in methods}

Generator = dMRI_Graph_Generator(str(data_root))

for subj in test_subjects:
    path = str(data_root / subj / "T1w" / "Diffusion_7T") + "/"
    print(f"\n=== Subject {subj} ===")
    Generator.load_data(path=path)

    r_mask = get_region_mask(Generator, region)
    print(f"  Region voxels: {r_mask.sum().item()}")

    # Sample N nodes from WHOLE region (no split)
    g = sample_region_graph(Generator, r_mask, N, k, perplexity, feature_mode)
    print(f"  Sampled: {g.normed_coords.shape[0]} nodes")

    coords_axis = g.normed_coords[:, split_axis].cpu().numpy()
    sp = float(np.median(coords_axis))

    # Border indices: A = lower side near boundary, B = upper side near boundary
    a_idx = np.where((coords_axis <  sp) & (coords_axis > sp - edge_margin))[0]
    b_idx = np.where((coords_axis >= sp) & (coords_axis < sp + edge_margin))[0]
    print(f"  split={sp:.4f}  border A={len(a_idx)}, B={len(b_idx)}")

    if len(a_idx) < 2 or len(b_idx) < 2:
        print("  Too few border nodes — skip")
        for m in methods:
            all_region_err[m].append(None)
            all_neighbor_err[m].append(None)
        continue

    xy_all = g.normed_coords.cpu().numpy()   # 3-D spatial coords
    xy_a, xy_b = xy_all[a_idx], xy_all[b_idx]

    # Embed FULL region with each method
    X = preprocess_for_baselines(g.normed_values.detach().cpu().numpy())
    emb_all = {}

    print("  NOFE  ...", end="\r")
    emb_all["nofe"] = model(g).detach().cpu().numpy()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

    print("  PCA   ...", end="\r")
    emb_all["pca"] = PCA(n_components=3).fit_transform(X)

    print("  t-SNE ...", end="\r")
    emb_all["tsne"] = TSNE(
        n_components=3, perplexity=min(30, g.normed_coords.shape[0] - 1),
        learning_rate="auto", random_state=42, method="exact"
    ).fit_transform(X)

    print("  UMAP  ...", end="\r")
    emb_all["umap"] = umap_lib.UMAP(n_components=3, random_state=42).fit_transform(X)

    for m in methods:
        E  = torch.from_numpy(emb_all[m])
        E_a, E_b = E[a_idx], E[b_idx]

        md_a   = mean_dist(E_a)
        md_b   = mean_dist(E_b)
        fl_a   = float(md_a) * 0.05
        fl_b   = float(md_b) * 0.05

        # A -> B
        _, nn_ab = cKDTree(xy_b).query(xy_a)
        cross_ab = torch.norm(E_a - E_b[nn_ab], dim=1)
        _, nn_aa = cKDTree(xy_a).query(xy_a, k=2)
        within_a = torch.norm(E_a - E_a[nn_aa[:, 1]], dim=1).clamp(min=fl_a)

        # B -> A
        _, nn_ba = cKDTree(xy_a).query(xy_b)
        cross_ba = torch.norm(E_b - E_a[nn_ba], dim=1)
        _, nn_bb = cKDTree(xy_b).query(xy_b, k=2)
        within_b = torch.norm(E_b - E_b[nn_bb[:, 1]], dim=1).clamp(min=fl_b)

        reg_all = torch.cat([cross_ab / md_a,     cross_ba / md_b])
        nbr_all = torch.cat([cross_ab / within_a, cross_ba / within_b])
        all_region_err[m].append(reg_all)
        all_neighbor_err[m].append(nbr_all)
        print(f"  {m:5s}: region={reg_all.mean():.4f}  neighbor={nbr_all.mean():.4f}")

---
## Results

In [ ]:
def cat_valid(lst):
    return torch.cat([x for x in lst if x is not None])


print("\n" + "="*60)
print(f"Region: {region}  |  split: {axis_label}")
print("="*60)
print("  Method   Region norm.(↓)         Neighbor norm.(↓)")
print("-"*58)
for m in methods:
    reg = cat_valid(all_region_err[m])
    nbr = cat_valid(all_neighbor_err[m])
    print(f"  {m:5s}    {reg.mean():.4f} ±{reg.std():.4f}         {nbr.mean():.4f} ±{nbr.std():.4f}")

In [ ]:
reg_m = [cat_valid(all_region_err[m]).mean().item()   for m in methods]
reg_s = [cat_valid(all_region_err[m]).std().item()    for m in methods]
nbr_m = [cat_valid(all_neighbor_err[m]).mean().item() for m in methods]
nbr_s = [cat_valid(all_neighbor_err[m]).std().item()  for m in methods]

colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]
x = np.arange(len(methods))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, means, stds, title in [
    (axes[0], reg_m, reg_s, " Region Norm. (↓ better)"),
    (axes[1], nbr_m, nbr_s, " Neighbor Norm. (↓ better)"),
]:
    bars = ax.bar(x, means, width=0.5, yerr=stds, capsize=4, color=colors, alpha=0.85)
    for bar, v in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(stds)*0.05,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([m.upper() for m in methods])
    ax.set_ylabel("Normalised boundary distance")
    ax.set_title(title, fontsize=11)

plt.suptitle(f"dMRI Patch Stitching — Region: {region}  |  split: {axis_label}", fontsize=13)
plt.tight_layout()
plt.savefig(Path.cwd() / f"region_{region}_results.png", dpi=150)
plt.show()
print(f"Saved → region_{region}_results.png")

In [ ]:
rows = []
for s_idx, subj in enumerate(test_subjects):
    row = {"subject": subj, "region": region, "split_axis": split_axis}
    for m in methods:
        r = all_region_err[m][s_idx]
        n = all_neighbor_err[m][s_idx]
        if r is not None:
            row[f"{m}_region_mean"]   = round(r.mean().item(), 4)
            row[f"{m}_region_std"]    = round(r.std().item(),  4)
            row[f"{m}_neighbor_mean"] = round(n.mean().item(), 4)
            row[f"{m}_neighbor_std"]  = round(n.std().item(),  4)
        else:
            for s in ["region_mean","region_std","neighbor_mean","neighbor_std"]:
                row[f"{m}_{s}"] = float("nan")
    rows.append(row)

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(Path.cwd() / f"region_{region}_detailed.csv", index=False)
print(f"Saved → region_{region}_detailed.csv")